# 04 — Engine Sensitivity Analysis: Tau vs. Tau_Review Trade-offs

This notebook evaluates the **sensitivity trade-offs** of NYAYA's expected-harm decision policy on held-out test data.

### Decision Policy Mechanics:
NYAYA balances the expected cost of fraudulent leakage ($H_F = P(\text{fraud}) \times \text{amount}$) against the expected humanitarian harm of blocking a genuine survivor ($H_H = (1 - P(\text{fraud})) \times \beta e^{\alpha S} u$):
$$H_\Delta = H_F - H_H$$

The engine disposition is governed by two thresholds:
- **$\tau$ (HOLD threshold)**: Outright hold if $P(\text{mule}) \ge \theta_{\text{mule}}$ or $H_\Delta \ge \tau$.
- **$\tau_{\text{review}}$ (REVIEW band)**: If not on HOLD, transactions falling in the uncertainty band (conformal set $\{ \text{legit}, \text{fraud} \}$ or $H_\Delta > -\tau_{\text{review}}$) and exceeding the survival cap (`stipend_cap`) have the stipend released immediately and the remainder escrowed for expedited review.
- **RELEASE**: All remaining transactions.

### Key Objectives:
1. **Load Sensitivity Results**: Ingest `artifacts/engine_tune.json` produced by `engine.tune` on held-out test transactions.
2. **Tabular Parameter Grid**: Display fraud leakage, fraud hold rate, legitimate hold/review rates, and population outcomes across $(\tau, \tau_{\text{review}})$ configurations.
3. **Trade-off Chart**: Plot **Fraud Leakage (₹ INR) vs. Legit Review Rate (%)**, with one curve per $\tau$ value.
4. **Displaced Victim Protection**: Evaluate the rate at which disaster victims are left with ₹0 (`victims_left_zero_rate`).
5. **Zone Rule Analysis**: Examine the impact of `zone_hold_needs_mule` on victim protection vs fraud leakage.
6. **Pareto Sweep**: If `artifacts/sim/pareto.json` is present, inspect multi-parameter sweeps and baseline comparisons.

*Note: All data and figures are read dynamically from whatever the files contain. No values are hand-set or hard-coded.*

In [ ]:
import sys
import os
from pathlib import Path

# Dataset / model tag
TAG = "v2"

# Add project root to sys.path
cwd = Path.cwd()
for candidate in [cwd, cwd.parent, Path("/home/ibm-user/shared/nyaya-kaustub")]:
    if (candidate / "models").is_dir() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))
        break

# Resolve data and artifacts directories with fallback
DATA_DIR = Path(os.environ.get("NYAYA_DATA", "/home/ibm-user/shared/data"))
if not DATA_DIR.exists():
    DATA_DIR = Path.cwd().parent / "data"

ARTIFACTS_DIR = Path(os.environ.get("NYAYA_ARTIFACTS", "/home/ibm-user/shared/nyaya/artifacts"))
if not ARTIFACTS_DIR.exists():
    ARTIFACTS_DIR = Path.cwd().parent / "artifacts"

print(f"Dataset / Model Tag:      {TAG}")
print(f"Project root on sys.path: {sys.path[0]}")
print(f"Data directory:           {DATA_DIR} (exists: {DATA_DIR.exists()})")
print(f"Artifacts directory:      {ARTIFACTS_DIR} (exists: {ARTIFACTS_DIR.exists()})")

In [ ]:
import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 220)

# Check matplotlib availability without installing
try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except ImportError:
    MATPLOTLIB_AVAILABLE = False
    print("Notice: matplotlib not present in environment. Outputting tabular summaries.")

## 1. Load Sensitivity Run Results (`artifacts/engine_tune.json`)
Ingest the sensitivity run generated by `python -m engine.tune`. This contains results evaluated on the held-out test split of Model A (seeded split unseen in training or calibration).

In [ ]:
tune_file = ARTIFACTS_DIR / "engine_tune.json"

if not tune_file.exists():
    print(f"Notice: {tune_file} does not exist yet.")
    print("To generate sensitivity results, run on the server:")
    print(f"    python -m engine.tune --data {DATA_DIR} --artifacts {ARTIFACTS_DIR} --tag {TAG}")
    tune_data = None
else:
    print(f"Loading sensitivity results from {tune_file}...")
    tune_data = json.loads(tune_file.read_text())
    print(f"Held-out transactions evaluated: {tune_data.get('rows_scored', 'N/A'):,}")
    if 'plan_default' in tune_data:
        print(f"Total fraud in held-out test set: ₹{tune_data['plan_default'].get('fraud_total_inr', 0):,.0f}")
    print(f"Policy defaults file: {list(tune_data.get('policy_file', {}).keys())}")

## 2. Parameter Grid Tabulation
Tabulate the $(\tau, \tau_{\text{review}})$ sensitivity grid, extracting fraud leakage, hold rates, review rates, and population outcomes dynamically from the file.

In [ ]:
if tune_data and "grid" in tune_data:
    df_grid = pd.DataFrame(tune_data["grid"])
    base_point = tune_data.get("plan_default", {})

    print(f"Sensitivity grid loaded: {len(df_grid)} parameter combinations.")
    print(f"Distinct tau values:        {sorted(df_grid['tau'].unique())}")
    print(f"Distinct tau_review values: {sorted(df_grid['tau_review'].unique())}")

    display_cols = [c for c in [
        "tau", "tau_review", "fraud_leakage_inr", "fraud_hold_rate",
        "legit_hold_rate", "legit_review_rate", "victims_left_zero_rate",
        "displaced_hold", "displaced_review", "ato_hold", "syndicate_hold"
    ] if c in df_grid.columns]

    print("\n--- Sensitivity Grid Summary ---")
    print(df_grid[display_cols].to_string(index=False))

    if base_point:
        print("\n--- Plan Production Default Baseline ---")
        base_df = pd.DataFrame([base_point])
        base_cols = [c for c in display_cols if c in base_df.columns]
        print(base_df[base_cols].to_string(index=False))
else:
    df_grid = None
    print("Sensitivity grid data not available. Awaiting engine_tune.json run.")

## 3. Trade-off Chart: Fraud Leakage vs. Legit Review Rate
Visualizes the primary operational trade-off:
- **X-axis**: Legit Review Rate (%)
- **Y-axis**: Fraud Leakage (₹ INR)
- **One curve per $\tau$**: Traces how adjusting $\tau_{\text{review}}$ shifts the balance between fraud prevention and review operational workload.
- **Plan Default Point**: Plotted as a reference marker.

In [ ]:
if df_grid is not None and MATPLOTLIB_AVAILABLE:
    plt.figure(figsize=(11, 6))

    unique_taus = sorted(df_grid["tau"].unique())
    for tau_val in unique_taus:
        sub = df_grid[df_grid["tau"] == tau_val].sort_values("tau_review")
        x_vals = sub["legit_review_rate"] * 100.0  # Percentage
        y_vals = sub["fraud_leakage_inr"]
        plt.plot(x_vals, y_vals, marker="o", linewidth=1.8, label=f"tau = {tau_val}")

        # Label points with tau_review value
        for _, row in sub.iterrows():
            tr = row["tau_review"]
            x = row["legit_review_rate"] * 100.0
            y = row["fraud_leakage_inr"]
            plt.annotate(f"tr={tr}", (x, y), textcoords="offset points", xytext=(4, 4), fontsize=8, alpha=0.75)

    # Plot baseline plan default
    if base_point:
        base_x = base_point.get("legit_review_rate", 0) * 100.0
        base_y = base_point.get("fraud_leakage_inr", 0)
        plt.scatter([base_x], [base_y], color="red", s=180, zorder=5, marker="*",
                    label=f"Plan Default (tau={base_point.get('tau')})")
        plt.annotate(f"Default (₹{base_y:,.0f})", (base_x, base_y),
                     textcoords="offset points", xytext=(-25, 12), fontweight="bold", color="red")

    plt.xlabel("Legitimate Transactions Sent to Review (%)", fontsize=11)
    plt.ylabel("Fraud Leakage (₹ INR)", fontsize=11)
    plt.title("Engine Sensitivity Trade-off: Fraud Leakage vs. Legit Review Rate (One line per tau)", fontsize=12)
    plt.grid(True, alpha=0.3)
    plt.legend(title="Thresholds", loc="upper right")
    plt.tight_layout()
    plt.show()
elif df_grid is not None:
    print("Tabular trade-off summary (Fraud Leakage vs. Legit Review Rate):")
    summary_tradeoff = df_grid[["tau", "tau_review", "legit_review_rate", "fraud_leakage_inr"]].sort_values(["tau", "tau_review"])
    print(summary_tradeoff.to_string(index=False))

## 4. Displaced Victim Protection and Zero-Release Rates
Examines humanitarian impact on victims in disaster zones (`displaced` population):
- `victims_left_zero_rate`: Fraction of displaced survivors whose emergency payments are withheld with ₹0 released immediately.
- `displaced_review`: Fraction receiving the immediate survival stipend while the remaining amount is escrowed.
- `displaced_hold`: Rate of complete hold for displaced victims.

In [ ]:
if df_grid is not None:
    pop_cols = [c for c in [
        "tau", "tau_review", "victims_left_zero_rate", "displaced_hold",
        "displaced_review", "ato_hold", "syndicate_hold"
    ] if c in df_grid.columns]

    print("--- Humanitarian Impact on Displaced Survivors Across Parameter Grid ---")
    print(df_grid[pop_cols].sort_values(["tau", "tau_review"]).to_string(index=False))

    if MATPLOTLIB_AVAILABLE and "victims_left_zero_rate" in df_grid.columns:
        plt.figure(figsize=(10, 5))
        for tau_val in unique_taus:
            sub = df_grid[df_grid["tau"] == tau_val].sort_values("tau_review")
            plt.plot(sub["tau_review"], sub["victims_left_zero_rate"] * 100.0, marker="s", label=f"tau = {tau_val}")

        plt.xlabel("tau_review (Rupees)", fontsize=11)
        plt.ylabel("Displaced Victims Left with ₹0 (%)", fontsize=11)
        plt.title("Displaced Victims Left with ₹0 Across tau and tau_review", fontsize=12)
        plt.grid(True, alpha=0.3)
        plt.legend(title="tau", loc="upper right")
        plt.tight_layout()
        plt.show()

## 5. The Trade-off the Zone Rule Makes (`grid_wide`)
Inspects the wider grid over `theta_mule` and `zone_hold_needs_mule`:
- Each point represents one $(\tau, \theta_{\text{mule}})$ setting with $\tau_{\text{review}} = 0$.
- Compares victim zero-release rate against fraud leakage percentage.
- Demonstrates how the disaster zone rule (requiring a mule indicator to freeze in-zone payments) prevents unnecessary harm to displaced victims.

In [ ]:
if tune_data and "grid_wide" in tune_data:
    df_wide = pd.DataFrame(tune_data["grid_wide"])
    df_wide["leak_pct"] = 100.0 * df_wide["fraud_leakage_inr"] / df_wide["fraud_total_inr"]

    sub_wide = df_wide[df_wide["tau_review"] == 0]

    if MATPLOTLIB_AVAILABLE:
        fig, ax = plt.subplots(figsize=(8, 5))
        for zone, color in [(False, "tab:blue"), (True, "tab:orange")]:
            s = sub_wide[sub_wide["zone_hold_needs_mule"] == zone]
            ax.scatter(100.0 * s["victims_left_zero_rate"], s["leak_pct"], c=color, label=f"zone rule {'on' if zone else 'off'}", alpha=0.8)

        chosen = tune_data.get("policy_file", {})
        ax.set_xlabel("Victims left with ₹0 (% of displaced payments)")
        ax.set_ylabel("Fraud leakage (% of fraud rupees)")
        ax.set_title(f"Policy grid, v2 held-out split ({tune_data.get('rows_scored', 0)} rows)")
        ax.legend()
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        print("Policy in force:", {k: chosen.get(k) for k in ["tau", "tau_review", "theta_mule", "zone_hold_needs_mule"]})
    else:
        print("Wide grid summary (tau_review == 0):")
        cols = ["tau", "theta_mule", "zone_hold_needs_mule", "victims_left_zero_rate", "leak_pct", "syndicate_hold"]
        print(sub_wide[[c for c in cols if c in sub_wide.columns]].head(20).to_string(index=False))
else:
    print("Wide grid (grid_wide) not present in tune_data.")

## 6. Pareto Sweep on Independent Suite (`sim.run pareto`)
When `artifacts/sim/pareto.json` is generated by Codex's evaluation suite, this cell loads the multi-parameter Pareto sweep around the policy in force and plots expected harm against baselines.

In [ ]:
pareto_file = ARTIFACTS_DIR / "sim" / "pareto.json"

if pareto_file.exists():
    par = json.loads(pareto_file.read_text())
    print("Suite:", par.get("suite_rows"), "rows from", par.get("suite_source"))
    rows = [{"parameter": p["parameter"], "value": p["value"],
             "victims_zero_rate": p["metrics"]["victims_left_with_zero_rate"],
             "fraud_leakage_rs": p["metrics"]["fraud_leakage_rupees"],
             "expected_harm_rs": p["metrics"]["total_expected_harm_rupees"],
             "loses_to": ", ".join(p.get("loses_to", [])) or "-"} for p in par.get("points", [])]
    pts = pd.DataFrame(rows)
    base = {k: v["total_expected_harm_rupees"] for k, v in par.get("baselines", {}).items()}
    print("Baseline expected harm (Rs):", {k: round(v) for k, v in base.items()})
    print("\nPareto Points Table:")
    print(pts.round(4).to_string(index=False))

    if MATPLOTLIB_AVAILABLE and "grids" in par:
        fig, axes = plt.subplots(1, len(par["grids"]), figsize=(4 * len(par["grids"]), 3.5), sharey=True)
        if len(par["grids"]) == 1:
            axes = [axes]
        for ax, (name, g) in zip(axes, pts.groupby("parameter")):
            ax.plot(g["value"], g["expected_harm_rs"] / 1e5, marker="o", label="NYAYA")
            for b, v in base.items():
                ax.axhline(v / 1e5, ls="--", lw=1, label=b)
            ax.set_title(name)
            ax.set_xlabel("Parameter Value")
        axes[0].set_ylabel("Total expected harm (Rs lakh)")
        axes[0].legend(fontsize=8)
        plt.tight_layout()
        plt.show()
else:
    print(f"Notice: {pareto_file} not found yet. It will be produced when `python -m sim.run pareto` is run.")